# 03 — Feature Assembly & Inspection

Loads all individual feature blocks and stacks them horizontally into the single combined matrix that will be fed into the KNN model. This notebook is primarily for inspection — it writes nothing to disk.

**What it does:**
- Loads all feature matrices from `data/features/` and expands them to the full album universe (inserting zero rows for albums that had no tags or ratings)
- Horizontally stacks tags, labels, types, and ratings into one matrix: shape ~2.2M albums × 6,521 features
- Analyses column sparsity to determine a safe pruning threshold — the minimum column density that removes noise columns without zeroing out any album that has at least one feature
- Identifies which albums have at least one non-zero feature (used as the recommendable set)

**Inputs:** `data/features/album_ids.pkl`, `data/features/album_tags_matrix.npz`, `data/features/album_labels_matrix.npz`, `data/features/album_types_matrix.npz`, `data/features/album_ratings_matrix.npz`, `data/mb_album.parquet`

**Outputs:** None — inspection only. The assembly logic is reproduced in `04-knn-training.ipynb` which writes the model artefacts.

**Run after:** `02-feature-ratings.ipynb` | **Run before:** `04-knn-training.ipynb`

## Imports and setup

Standard library imports plus the three sparse-matrix primitives from SciPy that underpin the whole pipeline:

- **`csr_matrix`** — Compressed Sparse Row format. Efficient for row slicing (looking up a single album's features) and matrix-vector products. This is the storage format used for every feature block and for the final combined matrix.
- **`hstack`** — Horizontally stacks sparse matrices side-by-side (concatenates columns). Used later to join the four feature blocks (tags, labels, types, ratings) into one wide matrix.
- **`load_npz` / `save_npz`** — Serialisation for SciPy sparse matrices. The `.npz` files on disk were written by the earlier feature-engineering notebooks.

`sklearn.preprocessing.normalize` is imported for potential L2 row-normalisation before KNN distance computation (cosine similarity requires unit-norm rows).

The `os.makedirs` call is a safety net — it ensures the features directory exists before anything tries to write into it.

In [ ]:
# ==============================================================================
# GLOBAL DEPENDENCIES & PIPELINE INITIALIZATION
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

# Core SciPy sparse matrix tools
from scipy.sparse import csr_matrix, hstack, save_npz, load_npz

# Visual analytics suite
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize

# Ensure the data architecture directory exists globally
os.makedirs('../data/features', exist_ok=True)

print("🚀 Master feature environment and sparse data paths initialized successfully.")

## Load feature blocks, expand to full album universe, and hstack

This cell does the core assembly work in four steps.

**Step 1 — Load the row index:** `album_ids.pkl` is an ordered list of MusicBrainz album UUIDs. It defines which row in each `.npz` file corresponds to which album. This list was produced by the earliest feature-engineering notebook and covers only albums that had *at least one tag* at the time it was built.

**Step 2 — Load the four feature blocks:** Each `.npz` is a sparse matrix with one row per album (in `album_id_order`) and columns representing individual feature dimensions:
- `X_tags` — one-hot / TF-IDF weighted user tags (3,041 columns)
- `X_labels` — one-hot encoded record labels (3,469 columns)
- `X_types` — one-hot encoded release types, e.g. Album / Single / EP (10 columns)
- `X_ratings` — normalised mean Listenbrainz rating (1 column)

**Step 3 — Expand to the full album universe:** The critical insight here is that `album_ids.pkl` was built from a filtered subset (albums with tags), so it covers ~1M albums. But `mb_album.parquet` contains the complete MusicBrainz catalogue of ~2.2M albums. If we skip this expansion step, albums without tags would simply be absent from the matrix — we could never recommend them or find neighbours for them. The `_expand` helper re-maps each matrix from its original row indices into a larger matrix of `n_full` rows, using `get_indexer` to look up where each original album lands in the full sorted index. Albums that had no features end up as all-zero rows.

**Step 4 — `hstack`:** `scipy.sparse.hstack` concatenates the four matrices column-wise into a single `(2,241,402 × 6,521)` matrix. This is the representation that gets fed to the KNN index. Each album is now described by all feature types simultaneously, and KNN distance is computed across all 6,521 dimensions at once.

In [3]:
# 1. Load the row index mappings
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# 2. Load all feature blocks from the folder
X_tags    = load_npz('../data/features/album_tags_matrix.npz')
X_labels  = load_npz('../data/features/album_labels_matrix.npz')
X_types   = load_npz('../data/features/album_types_matrix.npz')
X_ratings = load_npz('../data/features/album_ratings_matrix.npz')

# 3. Align all matrices to the full album universe.
#    album_ids.pkl may have been built from a filtered subset (e.g. only albums
#    with tags), so we expand each matrix to cover every album in mb_album.parquet,
#    inserting zero rows for albums that had no tags/labels/ratings.
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

if len(album_id_order) < len(full_album_ids):
    print(f"Expanding matrices from {len(album_id_order):,} → {len(full_album_ids):,} albums...")
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    X_tags    = _expand(X_tags,    current_pos, n_full)
    X_labels  = _expand(X_labels,  current_pos, n_full)
    X_types   = _expand(X_types,   current_pos, n_full)
    X_ratings = _expand(X_ratings, current_pos, n_full)
    album_id_order = full_album_ids.tolist()

# 4. Combine horizontally into the complete feature matrix
X_final_album_knn = hstack([X_tags, X_labels, X_types, X_ratings]).tocsr()

print(f"🚀 Matrix built out of data/features/ successfully!")
print(f"Final Model Dimensions: {X_final_album_knn.shape[0]:,} albums x {X_final_album_knn.shape[1]:,} features")

Expanding matrices from 1,008,102 → 2,241,402 albums...
🚀 Matrix built out of data/features/ successfully!
Final Model Dimensions: 2,241,402 albums x 6,521 features


## Print matrix shapes and nnz counts

A sanity-check print of the shape and **nnz** (number of non-zero elements) for every matrix.

**Why nnz matters:** Sparse matrices store only non-zero values, so nnz is the true measure of how much information a matrix contains. A very low nnz relative to `rows × cols` means the matrix is extremely sparse — most albums have no signal for most features. For KNN this is important: if two albums share very few non-zero columns, their cosine similarity will be near zero regardless of the values in those columns, making distance-based neighbours unreliable.

Reading the output:
- `X_tags` has 3M non-zeros across 3,041 columns — the densest block, averaging ~3 tags per album that has any tags at all.
- `X_labels` and `X_types` share the same nnz (402,047) because every labelled album has exactly one label *and* one type entry.
- `X_ratings` has only 44,334 non-zeros — Listenbrainz ratings are sparse relative to the full catalogue.
- The combined `X_final_album_knn` nnz is the sum of all blocks (3,853,425), confirming no information was lost in the hstack.

The `album_id_order` length check confirms the row index now covers all 2.2M albums after expansion.

In [4]:
import os

print(f"Working directory: {os.getcwd()}")
print()

matrices = {
    "X_tags":             X_tags,
    "X_labels":           X_labels,
    "X_types":            X_types,
    "X_ratings":          X_ratings,
    "X_final_album_knn":  X_final_album_knn,
}

for name, X in matrices.items():
    print(f"{name:25s}  shape={str(X.shape):25s}  nnz={X.nnz:,}")

print()
print(f"album_id_order length: {len(album_id_order):,}")

Working directory: /Users/niall/Desktop/ai_eng/mixtape/mixtape/features

X_tags                     shape=(2241402, 3041)            nnz=3,004,997
X_labels                   shape=(2241402, 3469)            nnz=402,047
X_types                    shape=(2241402, 10)              nnz=402,047
X_ratings                  shape=(2241402, 1)               nnz=44,334
X_final_album_knn          shape=(2241402, 6521)            nnz=3,853,425

album_id_order length: 2,241,402


## Compute per-column nnz stats

For every column in `X_final_album_knn`, count how many albums have a non-zero value in that column. This is the **column nnz** — a measure of how widely used each feature is across the catalogue.

A column with nnz = 1 means only one album in 2.2M carries that feature. Such columns contribute almost nothing to similarity comparisons because it's essentially impossible for two albums to share them. They also add noise: a KNN model treating rare tags as equally weighted features will occasionally find "neighbours" that share only an obscure tag with no other connection.

Computing `col_nnz = np.diff(X_csc.indptr)` is the efficient way to get this — converting to CSC (Compressed Sparse Column) format reorganises the data so that each column's non-zeros are stored contiguously, and `indptr` gives their start/end positions. `np.diff` over `indptr` then gives counts in O(n_columns) time without iterating over individual elements.

## Plot column nnz distribution

A histogram of column nnz values, typically displayed on a log scale because the distribution is highly skewed: a large number of columns appear in very few albums, while a small number of columns (e.g. common genre tags like "rock" or "electronic") appear in tens of thousands.

The plot is diagnostic — it answers "what does the long tail look like?" If 80% of columns have nnz < 10, pruning those columns removes very little real information while substantially reducing matrix width and therefore KNN computation cost. The x-axis threshold chosen here will directly inform the safe threshold analysis in later cells.

## Per-block breakdown table

Breaks the column nnz analysis down by feature block (tags, labels, types, ratings) rather than looking at the combined matrix as a whole. This matters because the blocks have very different sparsity profiles:

- **Tags** — long tail of rare tags (nnz = 1 for niche genres), but the top tags are very dense.
- **Labels** — most labels are small and release only a handful of albums; only major labels appear in many rows.
- **Types** — only 10 columns and extremely dense (every album with any metadata has a type), so none will be pruned.
- **Ratings** — single column, relatively sparse.

Seeing the breakdown helps verify that a global pruning threshold doesn't inadvertently wipe out an entire block. For example, if all label columns have nnz < 5, a threshold of 10 would eliminate the entire labels block — that would be a bug to catch here before it silently reaches training.

## CDF: columns retained vs min-nnz threshold

Plots the **cumulative distribution function** of column nnz values, reframed as "if I require every kept column to appear in at least N albums, how many columns survive?"

This lets you read off trade-offs directly. For example: "A threshold of 5 retains 60% of columns; a threshold of 20 retains 40% of columns." Lower thresholds keep more columns (more feature diversity) but include noisier, rarer features. Higher thresholds prune aggressively, reducing matrix width and KNN query time, but risk losing legitimate niche genre tags.

The CDF is more actionable than the histogram alone because it shows the cumulative impact of any given threshold choice, making it easy to identify the "elbow" where further tightening the threshold starts cutting into useful features.

## Analyse how many albums get zeroed at each threshold

The CDF above shows column retention — but the more critical question for recommendation quality is: **how many albums lose all their features if we prune at threshold N?**

An album gets zeroed out when every column it appears in gets dropped. This is the failure mode the safe threshold is designed to prevent: an album that had genuine features (say, one niche genre tag) becomes a zero-vector and KNN can no longer find meaningful neighbours for it — it would appear equally similar to every other album, or not at all depending on the query logic.

This cell sweeps over a range of threshold values and counts, for each one, how many albums would be reduced to all-zeros. The resulting curve shows the threshold at which albums start getting zeroed and how rapidly the damage grows. This is the key input to the safe threshold computation in the next cell.

## Compute has_features boolean mask

`has_features` is a boolean array of length 2,241,402 — one entry per album — where `True` means the album has at least one non-zero feature in the combined matrix.

This mask defines the **recommendable set**: the subset of albums the KNN model can actually do something useful with. Albums where `has_features = False` are all-zero rows; cosine similarity to any other album is undefined (division by zero), and nearest-neighbour search would return arbitrary results.

The mask is used in two ways:
1. **At query time** — filter out non-recommendable albums from results before returning them to the user. Even if KNN returns them as neighbours (which it shouldn't after column pruning), they should never surface as recommendations.
2. **For the Streamlit app selectbox** — only albums where `has_features = True` appear as valid query albums. This was the fix implemented in `93a5063` and is why the selectbox in the UI filters to recommendable albums only.

The count of `True` values here (~1M albums) is the effective catalogue size for recommendations.

## Compute safe threshold and prune columns

The **safe threshold** is the highest column-nnz cutoff that can be applied without zeroing out any album that currently has at least one feature (i.e. where `has_features = True`).

The algorithm works by finding, for each album that has features, the maximum nnz of any column it appears in — call this the album's "best column". The safe threshold is then `min(best_column_nnz) - 1` across all such albums: any threshold above this would drop the best (and only) column for at least one album, zeroing it out.

Why prune at all? Dropping low-nnz columns:
- Reduces matrix width from 6,521 to a smaller number, directly cutting KNN query time (approximate nearest-neighbour search cost scales with dimensionality).
- Removes features that are too rare to contribute meaningful signal — a tag shared by only 2 albums cannot help find good neighbours for most queries.
- Keeps memory footprint manageable when the pruned matrix is loaded into the NNDescent or FAISS index in `04-knn-training.ipynb`.

The safe threshold is a conservative choice — it guarantees zero collateral damage to the recommendable set. A more aggressive threshold could still be used if some album loss is acceptable (e.g. albums with only a single extremely rare tag may not be worth recommending anyway), but the safe threshold provides a principled lower bound.